# DSA03 – Natural Language Processing
**CO1 | Assessment Tool 1 – Application Based Questions**


In [1]:
import re


## Q1. Resume Information Extraction
Regex patterns extract name, email, mobile, skills and experience. A candidate is eligible if experience ≥ 2 years **and** Python is listed. `\b` word boundaries stop *JavaScript* from matching *Java*.

In [2]:
NAME_RE   = re.compile(r"^\s*name\s*[:\-]\s*([A-Za-z][A-Za-z .]+?)\s*$", re.I | re.M)
EMAIL_RE  = re.compile(r"\b[\w.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+\b")
# optional +91 / 91 / 0, then 10 digits starting 6-9 (spaces/hyphens allowed after 5 digits)
MOBILE_RE = re.compile(r"(?<!\d)(?:\+?91[\s-]?|0)?([6-9]\d{4}[\s-]?\d{5})(?!\d)")
SKILL_RE  = re.compile(r"\b(Python|Java|SQL|Machine\s+Learning|NLP)\b", re.I)
EXP_RE    = re.compile(r"(\d+(?:\.\d+)?)\s*\+?\s*(?:years?|yrs?)\b", re.I)

def extract_resume(text: str) -> dict:
    """Extract name, emails, mobiles, skills and experience from one resume."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Resume text is empty or invalid")
    name = NAME_RE.search(text)
    skills = sorted({re.sub(r"\s+", " ", s).title().replace("Nlp", "NLP").replace("Sql", "SQL")
                     for s in SKILL_RE.findall(text)})
    exp = EXP_RE.findall(text)
    return {
        "name": name.group(1) if name else "Not found",
        "emails": EMAIL_RE.findall(text),
        "mobiles": [re.sub(r"[\s-]", "", m) for m in MOBILE_RE.findall(text)],
        "skills": skills,
        "experience": max(map(float, exp)) if exp else 0.0,
    }

def is_eligible(profile: dict, min_exp: float = 2.0) -> bool:
    return profile["experience"] >= min_exp and "Python" in profile["skills"]

def run_q1():
    resumes = [
        """Name: Arjun Kumar
Email: arjun.kumar@gmail.com
Phone: +91 98765 43210
Skills: Python, SQL, Machine Learning, NLP
Experience: 3 years in data science""",
        """Name: Priya Sharma
Email: priya_s@outlook.com
Mobile: 9123456780
Skills: Java, SQL, JavaScript
Experience: 5 years as backend developer""",
        """Name: Rahul Menon
Email: rahul.menon@yahoo.in
Contact: 87654-32109
Skills: python, nlp
Experience: 1 year internship""",
        """Name: Divya R
Email: divya.r@company.co.in
Phone: 70011 22334
Skills: Python, Java, Machine Learning
Experience: 2.5 years""",
    ]
    print("=" * 60, "\nQ1. RESUME INFORMATION EXTRACTION\n" + "=" * 60)
    eligible = []
    for i, r in enumerate(resumes, 1):
        try:
            p = extract_resume(r)
        except ValueError as e:
            print(f"Resume {i}: {e}"); continue
        print(f"\n--- Candidate Profile {i} ---")
        print(f"Name       : {p['name']}")
        print(f"Email(s)   : {', '.join(p['emails']) or 'None'}")
        print(f"Mobile(s)  : {', '.join(p['mobiles']) or 'None'}")
        print(f"Skills     : {', '.join(p['skills']) or 'None'}")
        print(f"Experience : {p['experience']} year(s)")
        ok = is_eligible(p)
        print(f"Eligible   : {'YES' if ok else 'NO'}")
        if ok: eligible.append(p["name"])
    print("\nShortlisted (>=2 yrs & Python):", ", ".join(eligible) or "None")

In [3]:
run_q1()

Q1. RESUME INFORMATION EXTRACTION

--- Candidate Profile 1 ---
Name       : Arjun Kumar
Email(s)   : arjun.kumar@gmail.com
Mobile(s)  : 9876543210
Skills     : Machine Learning, NLP, Python, SQL
Experience : 3.0 year(s)
Eligible   : YES

--- Candidate Profile 2 ---
Name       : Priya Sharma
Email(s)   : priya_s@outlook.com
Mobile(s)  : 9123456780
Skills     : Java, SQL
Experience : 5.0 year(s)
Eligible   : NO

--- Candidate Profile 3 ---
Name       : Rahul Menon
Email(s)   : rahul.menon@yahoo.in
Mobile(s)  : 8765432109
Skills     : NLP, Python
Experience : 1.0 year(s)
Eligible   : NO

--- Candidate Profile 4 ---
Name       : Divya R
Email(s)   : divya.r@company.co.in
Mobile(s)  : 7001122334
Skills     : Java, Machine Learning, Python
Experience : 2.5 year(s)
Eligible   : YES

Shortlisted (>=2 yrs & Python): Arjun Kumar, Divya R


## Q2. Product Search System
Exact (`\bkw\b`), prefix (`\bkw\w*`), suffix (`\w*kw\b`) and partial (`kw`) searches, all case-insensitive, with a match-count report.

In [3]:
PRODUCTS = ["Laptop", "Laptop Bag", "Gaming Laptop", "Smartphone", "Phone Case",
            "Wireless Headphones", "Headphone Stand", "Bluetooth Speaker",
            "Smart Watch", "Watch Strap", "Mouse Pad", "Wireless Mouse",
            "Keyboard", "Mechanical Keyboard", "Tablet", "Tablet Cover"]

def build_pattern(keyword: str, mode: str) -> re.Pattern:
    """Build a case-insensitive regex for the given search mode."""
    k = re.escape(keyword.strip())
    patterns = {
        "exact":   rf"\b{k}\b",        # whole word
        "prefix":  rf"\b{k}\w*",       # word starts with keyword
        "suffix":  rf"\w*{k}\b",       # word ends with keyword
        "partial": rf"{k}",            # keyword anywhere
    }
    if mode not in patterns:
        raise ValueError(f"Unknown search mode: {mode}")
    return re.compile(patterns[mode], re.I)

def search_products(keyword: str, mode: str, products=PRODUCTS) -> list:
    if not keyword or not keyword.strip():
        raise ValueError("Search keyword cannot be empty")
    pat = build_pattern(keyword, mode)
    return [p for p in products if pat.search(p)]

def run_q2():
    print("\n" + "=" * 60, "\nQ2. PRODUCT SEARCH SYSTEM\n" + "=" * 60)
    tests = [("laptop", "exact"), ("wire", "prefix"), ("board", "suffix"),
             ("phone", "partial"), ("WATCH", "exact"), ("", "exact"), ("xyz", "partial")]
    report = []
    for kw, mode in tests:
        try:
            res = search_products(kw, mode)
        except ValueError as e:
            print(f"\nSearch ('{kw}', {mode}): ERROR - {e}"); continue
        print(f"\nSearch: '{kw}' | Mode: {mode}")
        print("  Matches:", res if res else "No products found")
        report.append((kw, mode, len(res)))
    print("\n--- Search Report ---")
    print(f"{'Keyword':<10}{'Mode':<10}{'Matches':>8}")
    for kw, mode, n in report:
        print(f"{kw:<10}{mode:<10}{n:>8}")

In [4]:
run_q2()


Q2. PRODUCT SEARCH SYSTEM

Search: 'laptop' | Mode: exact
  Matches: ['Laptop', 'Laptop Bag', 'Gaming Laptop']

Search: 'wire' | Mode: prefix
  Matches: ['Wireless Headphones', 'Wireless Mouse']

Search: 'board' | Mode: suffix
  Matches: ['Keyboard', 'Mechanical Keyboard']

Search: 'phone' | Mode: partial
  Matches: ['Smartphone', 'Phone Case', 'Wireless Headphones', 'Headphone Stand']

Search: 'WATCH' | Mode: exact
  Matches: ['Smart Watch', 'Watch Strap']

Search ('', exact): ERROR - Search keyword cannot be empty

Search: 'xyz' | Mode: partial
  Matches: No products found

--- Search Report ---
Keyword   Mode       Matches
laptop    exact            3
wire      prefix           2
board     suffix           2
phone     partial          4
WATCH     exact            2
xyz       partial          0


## Q3. University Registration Validation
One regex per field (register number, email, course code, semester, mobile). The registration succeeds only if every field is valid.

| Field | Rule |
|---|---|
| Register No. | 9 digits starting with 19 |
| Email | @saveetha.com or @simats.edu.in |
| Course code | 3 capital letters + 2 digits |
| Semester | 1–8, optional 'Semester' prefix |
| Mobile | 10 digits starting 6–9, optional +91 |

In [5]:
RULES = {
    # 9 digits: starts with 19 (e.g. 192424001)
    "Register Number": (re.compile(r"^19\d{7}$"), "9 digits starting with 19"),
    # institutional domain only
    "Email": (re.compile(r"^[A-Za-z0-9._%+-]+@(?:saveetha\.com|simats\.edu\.in)$", re.I),
              "name@saveetha.com or name@simats.edu.in"),
    # 3 capital letters + 2 digits (e.g. DSA03)
    "Course Code": (re.compile(r"^[A-Z]{3}\d{2}$"), "3 capital letters + 2 digits, e.g. DSA03"),
    # 1-8, optionally prefixed by 'Sem' / 'Semester'
    "Semester": (re.compile(r"^(?:sem(?:ester)?\s*)?[1-8]$", re.I), "1 to 8 (e.g. 3 or Semester 3)"),
    # Indian mobile, optional +91
    "Mobile": (re.compile(r"^(?:\+91[\s-]?)?[6-9]\d{9}$"), "10 digits starting 6-9, optional +91"),
}

def validate_student(record: dict) -> bool:
    all_ok = True
    for field, (pat, hint) in RULES.items():
        value = str(record.get(field, "")).strip()
        if pat.fullmatch(value):
            print(f"  [VALID]   {field}: {value}")
        else:
            print(f"  [INVALID] {field}: '{value}'  -> expected {hint}")
            all_ok = False
    return all_ok

def run_q3():
    print("\n" + "=" * 60, "\nQ3. UNIVERSITY REGISTRATION VALIDATION\n" + "=" * 60)
    students = [
        {"Register Number": "192424001", "Email": "john@saveetha.com", "Course Code": "DSA03",
         "Semester": "Semester 3", "Mobile": "+91 9876543210"},
        {"Register Number": "29242400", "Email": "mary@gmail.com", "Course Code": "dsa3",
         "Semester": "9", "Mobile": "1234567890"},
    ]
    for i, s in enumerate(students, 1):
        print(f"\nStudent {i}:")
        ok = validate_student(s)
        print(f"  >> REGISTRATION STATUS: {'SUCCESSFUL' if ok else 'FAILED'}")

In [7]:
run_q3()


Q3. UNIVERSITY REGISTRATION VALIDATION

Student 1:
  [VALID]   Register Number: 192424001
  [VALID]   Email: john@saveetha.com
  [VALID]   Course Code: DSA03
  [VALID]   Semester: Semester 3
  [VALID]   Mobile: +91 9876543210
  >> REGISTRATION STATUS: SUCCESSFUL

Student 2:
  [INVALID] Register Number: '29242400'  -> expected 9 digits starting with 19
  [INVALID] Email: 'mary@gmail.com'  -> expected name@saveetha.com or name@simats.edu.in
  [INVALID] Course Code: 'dsa3'  -> expected 3 capital letters + 2 digits, e.g. DSA03
  [INVALID] Semester: '9'  -> expected 1 to 8 (e.g. 3 or Semester 3)
  [INVALID] Mobile: '1234567890'  -> expected 10 digits starting 6-9, optional +91
  >> REGISTRATION STATUS: FAILED


## Conclusion
Regular expressions gave a compact way to extract information (Q1), match flexible search patterns (Q2) and validate structured input (Q3). Each module was tested with valid, invalid and edge-case inputs and the outputs match expectations.